# Can we predict arrears next month?
Target: `arrears_next_month`. Each row is an account-month; accounts repeat across months and the model is meant to score *future* months, so the split is by time **and** by whole accounts (checklist 1-2).

**Leakage note (checklist 7):** `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time. It is almost a restatement of the target, so it is dropped. `meter_ref` is an identifier and is used only for grouping; `billing_month` is used only for splitting.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, brier_score_loss

df = pd.read_csv('data.csv', parse_dates=['billing_month'])
TARGET = 'arrears_next_month'
print(df.shape, 'base rate', df[TARGET].mean().round(4))
print(df.groupby('final_notice_sent')[TARGET].mean())

## 1. Test set, set aside before any modelling
Latest 4 billing months, restricted to a random 20% of accounts held out entirely. Training rows are earlier months of the other accounts only, so no account and no later period leaks into training.

In [ ]:
rng = np.random.default_rng(0)
accounts = df.meter_ref.unique()
test_accounts = set(rng.choice(accounts, size=int(0.2 * len(accounts)), replace=False))
months = np.sort(df.billing_month.unique())
cutoff = months[-4]
is_test_acct = df.meter_ref.isin(test_accounts)
test = df[is_test_acct & (df.billing_month >= cutoff)].copy()
train = df[~is_test_acct & (df.billing_month < cutoff)].copy().reset_index(drop=True)
assert set(train.meter_ref).isdisjoint(test.meter_ref) and train.billing_month.max() < test.billing_month.min()
print(len(train), len(test), 'train rate', train[TARGET].mean().round(4), 'test rate', test[TARGET].mean().round(4))

## 2. Candidates (named before scoring) and features
Features (known at prediction time): `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

Candidates:
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: standardised numerics + one-hot tariff, L2 with C in {0.1, 1, 10}.
3. `hgb`: gradient boosting with max_depth in {2, 4}, 200 iterations, lr 0.05 (small, fixed grid).

Selection metric: mean PR-AUC across CV folds (the positive class is ~12%). Threshold: the value maximising F1 on pooled out-of-fold predictions of the chosen model.

In [ ]:
NUM = ['occupants', 'kwh', 'bill_eur', 'late_payments_12m']; CAT = ['tariff']; FEATS = NUM + CAT


def pre(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', NUM),
                              ('c', OneHotEncoder(handle_unknown='ignore'), CAT)])


candidates = {'baseline': make_pipeline(pre(False), DummyClassifier(strategy='prior'))}
for C_ in [0.1, 1, 10]:
    candidates[f'logreg_C{C_}'] = make_pipeline(pre(True), LogisticRegression(C=C_, max_iter=1000))
for d in [2, 4]:
    candidates[f'hgb_d{d}'] = make_pipeline(pre(False), HistGradientBoostingClassifier(max_depth=d, learning_rate=0.05, max_iter=200, random_state=0))
list(candidates)

## 3. Cross-validation inside training rows
Expanding-window over time (5 consecutive month blocks as validation, after an initial burn-in). For each fold the validation rows are a random third of accounts in that block; training rows are earlier months from the *other* accounts. All preprocessing lives in the pipeline and is fit per fold.

In [ ]:
tm = np.sort(train.billing_month.unique())
blocks = np.array_split(tm, 8)[3:]
accts = train.meter_ref.unique()


def folds():
    r = np.random.default_rng(1)
    for b in blocks:
        val_accts = set(r.choice(accts, size=len(accts) // 3, replace=False))
        in_val = train.meter_ref.isin(val_accts)
        va = train.index[in_val & train.billing_month.isin(b)]
        tr = train.index[~in_val & (train.billing_month < b.min())]
        yield tr, va


FOLDS = list(folds())
print([(len(a), len(b)) for a, b in FOLDS])
oof = {}; rows = []
for name, m in candidates.items():
    p = pd.Series(np.nan, index=train.index); s = []
    for tr, va in FOLDS:
        mm = clone(m).fit(train.loc[tr, FEATS], train.loc[tr, TARGET])
        pv = mm.predict_proba(train.loc[va, FEATS])[:, 1]; p[va] = pv
        s.append((average_precision_score(train.loc[va, TARGET], pv), roc_auc_score(train.loc[va, TARGET], pv)))
    oof[name] = p; s = np.array(s)
    rows.append((name, s[:, 0].mean(), s[:, 0].std(), s[:, 1].mean()))
cv = pd.DataFrame(rows, columns=['model', 'pr_auc_mean', 'pr_auc_sd', 'roc_auc_mean']).sort_values('pr_auc_mean', ascending=False)
cv

## 4. Choose model and threshold on CV only

In [ ]:
best = cv.iloc[0].model
mask = oof[best].notna(); y_oof = train.loc[mask, TARGET]; p_oof = oof[best][mask]
ths = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_oof, p_oof >= t) for t in ths]
THRESH = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THRESH, 2), '| CV F1:', round(max(f1s), 3))

**Decision recorded before touching the test set:** the model and threshold printed above are final. No further tuning follows.

## 5. Refit on all training rows, score the test set once

In [ ]:
final = clone(candidates[best]).fit(train[FEATS], train[TARGET])
pt = final.predict_proba(test[FEATS])[:, 1]; yt = test[TARGET]; pred = pt >= THRESH
base = clone(candidates['baseline']).fit(train[FEATS], train[TARGET]).predict_proba(test[FEATS])[:, 1]
res = pd.DataFrame({'chosen': [roc_auc_score(yt, pt), average_precision_score(yt, pt), precision_score(yt, pred), recall_score(yt, pred), f1_score(yt, pred), brier_score_loss(yt, pt)],
                    'baseline': [roc_auc_score(yt, base), average_precision_score(yt, base), np.nan, np.nan, np.nan, brier_score_loss(yt, base)]},
                   index=['ROC-AUC', 'PR-AUC', 'precision@thr', 'recall@thr', 'F1@thr', 'Brier'])
print('test rows', len(test), 'positives', int(yt.sum())); res.round(3)

## Conclusion
See the table above for the headline held-out score, read against the baseline column (a no-skill PR-AUC equals the base rate) and the CV table. Caveats: `final_notice_sent` was excluded because it is unavailable at prediction time (including it would give near-perfect but unusable scores); the test set covers only 4 months and ~20% of accounts, so estimates carry sampling noise.